# 38. 유니온 파인드와 최소 신장 트리

경시 · 문제 186–190

**사용법**
1. 바로 아래 **🔧 채점기 준비** 셀을 먼저 실행합니다(▶).
2. 각 문제의 `%%writefile` 셀에 코드를 작성하고 실행하면 파일로 저장됩니다.
3. 그 아래 `check(번호)` 셀을 실행하면 채점됩니다.
4. 직접 입력을 넣어 실행해 보려면 새 셀에 `%run p001.py`처럼 입력합니다.

> 정답과 해설은 위키독스 본문의 접힌 상자에 있습니다. 충분히 고민한 뒤에 펼쳐 보세요.

> 제3부의 시간 제한은 Colab 기준입니다. 숨은 테스트에는 제한 범위의 큰 입력이 들어 있으므로, 효율이 낮은 풀이는 '시간 초과'가 납니다.

In [ ]:
#@title 🔧 채점기 준비 (먼저 실행하세요)
TESTS = {186: [['5 6\n0 1 2\n1 1 2\n1 2 3\n0 3 2\n1 1 3\n1 4 5', 'YES\nNO\nYES\nNO', True, None], ['1 1\n1 1 1', 'YES', False, None], ['3 2\n0 1 1\n1 1 2', 'NO', False, None], [None, 'sha256:dd875ac3160620c7ec23b4e5406a67774b7975dc316c2a3b78f6b84ffd3dc36a', False, 'import random\nrandom.seed(218)\nn = q = 200000\nprint(n, q)\nfor _ in range(q):\n    print(random.randint(0, 1), random.randint(1, n), random.randint(1, n))'], [None, 'sha256:64dd52e165b1139b7b1ef4c41a8e0271810707fa3b339871c6b8db4f1b691f91', False, 'import random\nrandom.seed(219)\nn = q = 200000\nprint(n, q)\nfor i in range(1, q // 2 + 1):\n    print(0, i, i + 1)\nfor _ in range(q - q // 2):\n    print(1, 1, random.randint(1, n))']], 187: [['4 5\n1 2 3\n2 3 1\n1 3 2\n3 4 5\n2 4 4', '7', True, None], ['3 1\n1 2 1', '-1', True, None], ['1 0', '0', False, None], ['2 2\n1 2 5\n1 2 3', '3', False, None], [None, '28602054004', False, 'import random\nrandom.seed(220)\nn, m = 100000, 200000\nprint(n, m)\nfor v in range(2, n + 1):\n    print(random.randint(1, v - 1), v, random.randint(1, 10**6))\nfor _ in range(m - n + 1):\n    print(random.randint(1, n), random.randint(1, n), random.randint(1, 10**6))']], 188: [['4\nkim lee\nlee park\nchoi han\npark kim', '2\n3\n2\n3', True, None], ['1\na a', '1', False, None], ['3\na b\nc d\nb c', '2\n2\n4', False, None], [None, 'sha256:535b8304449e0237274f287828d20478151116e78e9619cebd24b622c7d64bf7', False, "import random\nrandom.seed(221)\nf = 100000\nprint(f)\nfor _ in range(f):\n    print(f'u{random.randrange(60000)}', f'u{random.randrange(60000)}')"]], 189: [['6 5\n0 1\n1 2\n2 3\n5 4\n0 4', '0', True, None], ['6 5\n0 1\n1 2\n1 3\n0 3\n4 5', '4', True, None], ['3 3\n0 1\n1 2\n2 0', '3', False, None], ['3 2\n0 1\n0 1', '2', False, None], [None, '500000', False, 'import random\nrandom.seed(222)\nn, m = 500000, 1000000\nprint(n, m)\nperm = list(range(n))\nrandom.shuffle(perm)\nfor i in range(1, n):\n    print(perm[random.randint(0, i - 1)], perm[i])\nfor _ in range(m - n + 1):\n    a, b = random.sample(range(n), 2)\n    print(a, b)'], [None, '0', False, 'import random\nrandom.seed(223)\nn, m = 500000, 499999\nprint(n, m)\nfor i in range(1, n):\n    print(i - 1, i)']], 190: [['5\n11 -15 -15\n14 -5 -15\n-1 -1 -5\n10 -4 -1\n19 -4 19', '4', True, None], ['1\n0 0 0', '0', False, None], ['2\n1 100 1000\n2 200 2000', '1', False, None], ['3\n0 0 0\n0 0 0\n5 5 5', '5', False, None], [None, '427012126', False, 'import random\nrandom.seed(224)\nn = 100000\nprint(n)\nfor _ in range(n):\n    print(random.randint(-10**9, 10**9), random.randint(-10**9, 10**9), random.randint(-10**9, 10**9))']]}
TL = {186: 3, 187: 3, 188: 3, 189: 4, 190: 4}
import subprocess, sys, pathlib, hashlib

_GEN_CACHE = {}

def _norm(s):
    return "\n".join(l.rstrip() for l in s.strip("\n").splitlines()).strip("\n")

def _input_of(inp, gen):
    if gen is None:
        return inp
    if gen not in _GEN_CACHE:
        _GEN_CACHE[gen] = subprocess.run([sys.executable, "-c", gen], capture_output=True, text=True).stdout
    return _GEN_CACHE[gen]

def check(pid):
    """p{번호}.py 를 모든 테스트로 실행해 채점합니다."""
    path = pathlib.Path(f"p{pid:03d}.py")
    if not path.exists():
        print(f"먼저 위의 %%writefile p{pid:03d}.py 셀을 실행하세요."); return
    limit = TL[pid]
    ok = 0
    for i, (inp, exp, shown, gen) in enumerate(TESTS[pid], 1):
        tag = f"예제 {i}" if shown else f"숨은 테스트 {i}"
        data = _input_of(inp, gen)
        try:
            r = subprocess.run([sys.executable, str(path)], input=data.rstrip("\n") + "\n",
                               capture_output=True, text=True, timeout=limit)
        except subprocess.TimeoutExpired:
            print(f"⏱ {tag}: 시간 초과 ({limit}초)"); continue
        got = _norm(r.stdout)
        if exp.startswith("sha256:"):
            got_cmp = "sha256:" + hashlib.sha256(got.encode()).hexdigest()
        else:
            got_cmp = got
        if r.returncode:
            last = r.stderr.strip().splitlines()[-1] if r.stderr.strip() else "알 수 없는 오류"
            print(f"💥 {tag}: 실행 오류 → {last}")
        elif got_cmp == exp:
            ok += 1; print(f"✅ {tag}")
        elif shown:
            print(f"❌ {tag}\n   입력: {inp!r}\n   기대: {exp!r}\n   실제: {got!r}")
        elif gen is not None:
            print(f"❌ {tag}  (큰 입력)")
        else:
            print(f"❌ {tag}  (입력: {inp[:60]!r})")
    total = len(TESTS[pid])
    print("─" * 30)
    print(f"🎉 {pid:03d}번 통과! ({ok}/{total})" if ok == total else f"{ok}/{total} 통과 — 다시 도전해 보세요.")
print('채점기 준비 완료!')

## 186. 합치고 묻기

1번부터 N번까지의 원소가 처음에는 각자 다른 집합에 있습니다. Q개의 명령을 처리하십시오. `0 a b`는 a가 속한 집합과 b가 속한 집합을 합치고, `1 a b`는 a와 b가 같은 집합이면 `YES`, 아니면 `NO`를 출력합니다.

**입력** 첫째 줄에 N과 Q, 이어서 Q줄에 명령이 주어집니다.

**출력** `1` 명령마다 한 줄에 답을 출력합니다.

**제한** 1 ≤ N ≤ 2×10⁵, 1 ≤ Q ≤ 2×10⁵, 시간 제한 3초

**예제 1**
입력
```
5 6
0 1 2
1 1 2
1 2 3
0 3 2
1 1 3
1 4 5
```
출력
```
YES
NO
YES
NO
```

In [ ]:
%%writefile p186.py
# 여기에 코드를 작성하세요


In [ ]:
check(186)

## 187. 크루스칼의 다리

N개의 섬과 M개의 다리 후보(양방향, 비용)가 주어집니다. 모든 섬이 이어지도록 다리를 고를 때 비용 합의 최솟값을 구하십시오. 모든 섬을 이을 수 없으면 −1을 출력합니다.

**입력** 첫째 줄에 N과 M, 이어서 M줄에 a b w가 주어집니다.

**출력** 최소 비용 또는 −1을 출력합니다.

**제한** 1 ≤ N ≤ 10⁵, 0 ≤ M ≤ 2×10⁵, 1 ≤ w ≤ 10⁶, 시간 제한 3초

**예제 1**
입력
```
4 5
1 2 3
2 3 1
1 3 2
3 4 5
2 4 4
```
출력
```
7
```

**예제 2**
입력
```
3 1
1 2 1
```
출력
```
-1
```

In [ ]:
%%writefile p187.py
# 여기에 코드를 작성하세요


In [ ]:
check(187)

## 188. 친구의 친구

친구 관계가 하나씩 F개 주어집니다. 관계 `a b`가 맺어질 때마다 a가 속한 친구 무리(친구의 친구… 로 이어진 모든 사람)의 크기를 출력하십시오. 이름은 영어 소문자입니다.

**입력** 첫째 줄에 F, 이어서 F줄에 두 사람의 이름이 주어집니다.

**출력** 관계마다 한 줄에 무리의 크기를 출력합니다.

**제한** 1 ≤ F ≤ 10⁵, 이름 길이 ≤ 20, 시간 제한 3초

**예제 1**
입력
```
4
kim lee
lee park
choi han
park kim
```
출력
```
2
3
2
3
```

In [ ]:
%%writefile p188.py
# 여기에 코드를 작성하세요


In [ ]:
check(188)

## 189. 처음 생긴 순환

N개의 정점에 무방향 간선 M개가 차례로 추가됩니다. 처음으로 순환이 생기는 것은 몇 번째 간선을 추가했을 때인지 출력하십시오. 끝까지 순환이 생기지 않으면 0을 출력합니다.

**입력** 첫째 줄에 N과 M, 이어서 M줄에 간선의 두 끝점이 주어집니다(0부터 N−1).

**출력** 처음 순환을 만든 간선의 번호(1부터) 또는 0을 출력합니다.

**제한** 3 ≤ N ≤ 5×10⁵, 1 ≤ M ≤ 10⁶, 자기 자신으로 가는 간선은 없음, 시간 제한 4초

**예제 1**
입력
```
6 5
0 1
1 2
2 3
5 4
0 4
```
출력
```
0
```

**예제 2**
입력
```
6 5
0 1
1 2
1 3
0 3
4 5
```
출력
```
4
```

In [ ]:
%%writefile p189.py
# 여기에 코드를 작성하세요


In [ ]:
check(189)

## 190. 우주 정거장 연결

3차원 공간에 N개의 정거장이 있습니다. 두 정거장 (x₁, y₁, z₁), (x₂, y₂, z₂)를 터널로 잇는 비용은 min(|x₁−x₂|, |y₁−y₂|, |z₁−z₂|)입니다. 모든 정거장을 잇는 최소 비용을 구하십시오.

**입력** 첫째 줄에 N, 이어서 N줄에 정거장의 좌표가 주어집니다.

**출력** 최소 비용을 출력합니다.

**제한** 1 ≤ N ≤ 10⁵, |좌표| ≤ 10⁹, 시간 제한 4초

**예제 1**
입력
```
5
11 -15 -15
14 -5 -15
-1 -1 -5
10 -4 -1
19 -4 19
```
출력
```
4
```

In [ ]:
%%writefile p190.py
# 여기에 코드를 작성하세요


In [ ]:
check(190)